## Querying PostgreSQL via SDK

### Installing Utilities and Libraries

In [1]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 python-dotenv

   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   -------------- ------------------------- 1.3/3.7 MB 6.8 MB/s eta 0:00:01
   ---------------------------- ----------- 2.6/3.7 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 6.3 MB/s  0:00:00

   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------- ----------------------------- 1/4 [psycopg_pool]
   -------------------- ------------------- 2/4 [psycopg-binary]
   ------------------------------ --------- 3/4 [psycopg]
   ---


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
host = os.getenv("DATABASE_HOSTNAME")
db_name = os.getenv("DATABASE_NAME")
username = os.getenv("DATABASE_USERNAME")
password = os.getenv("DATABASE_PASSWORD")

### Create a Connection Pool

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Use the Connection Pool to Execute a SELECT Query

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT *
            FROM ESG.Companies
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

### Filter Data Using WHERE

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT *
        FROM ESG.EmissionRecords
        WHERE Scope = %s
        """,
        (1,))

        results = cur.fetchall()

        for row in results:
            print(row)

### Filter Emissions Greater than 500

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT *
        FROM ESG.EmissionRecords
        WHERE CO2_Emissions > %s
        """,
        (500,))

        results = cur.fetchall()

        for row in results:
            print(row)

### Sort by Emissions

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT *
            FROM ESG.EmissionRecords
            ORDER BY CO2_Emissions ASC
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

### Calculate Total Emissions per Company

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                CompanyID,
                SUM(CO2_Emissions) AS TotalEmissions
            FROM ESG.EmissionRecords
            GROUP BY CompanyID
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

### Get Pool Statistics

In [ ]:
print(
    pool.get_stats()
)